# Transcript features, prompt disagreement, dataset joins

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
CONFIG = PROJECT / "configs/linguistic_features_v1.json"
OUTPUT = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
REPORT = PROJECT / "reports/step04_linguistic_extraction_qc.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED_ASR_SHA = (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_ASR_SHA

# Preserve apostrophes inside words, and normalize case.
# This is a simple lexical tokenizer, not linguistic parsing.
TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

feature_names = [
    "word_count",
    "unique_word_count",
    "type_token_ratio",
    "mean_word_length",
    "character_count",
    "adjacent_repeat_count",
    "adjacent_repeat_ratio"
]

feature_config = {
    "name": "linguistic_features_v1",
    "status": "proposed_not_frozen",
    "source_asr_sha256": EXPECTED_ASR_SHA,
    "normalization": "unicode_casefold",
    "tokenization": "unicode_words_internal_apostrophes",
    "punctuation": "excluded_from_token_counts",
    "features": feature_names,
    "empty_text_policy": "zero_counts_nan_ratios",
    "notes": [
        "Features describe ASR output, not verified speech.",
        "No prompt or fluency label is used.",
        "TTR is dependent on utterance length.",
        "Adjacent repetition is a transcript proxy."
    ]
}

CONFIG.parent.mkdir(parents=True, exist_ok=True)

if CONFIG.exists():
    existing = json.loads(CONFIG.read_text(encoding="utf-8"))
    assert existing == feature_config, (
        "Existing linguistic config differs; not overwriting."
    )
else:
    CONFIG.write_text(
        json.dumps(feature_config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

asr = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(asr) == 2000
assert asr["audio_id"].nunique() == 400
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert asr["transcription"].str.strip().ne("").all()

def extract_features(text):
    words = [
        word.casefold()
        for word in TOKEN_PATTERN.findall(str(text))
    ]

    n = len(words)
    unique_n = len(set(words))
    chars = sum(len(word) for word in words)

    repeats = sum(
        words[i] == words[i-1]
        for i in range(1, n)
    )

    return {
        "word_count": n,
        "unique_word_count": unique_n,
        "type_token_ratio": unique_n / n if n else np.nan,
        "mean_word_length": chars / n if n else np.nan,
        "character_count": chars,
        "adjacent_repeat_count": repeats,
        "adjacent_repeat_ratio": (
            repeats / (n - 1) if n > 1 else np.nan
        )
    }

extracted = pd.DataFrame(
    [extract_features(t) for t in asr["transcription"]]
)

result = pd.concat(
    [
        asr[
            ["pilot_index", "audio_id", "speaker", "hypothesis_id"]
        ].reset_index(drop=True),
        extracted
    ],
    axis=1
)

assert len(result) == 2000
assert result["audio_id"].nunique() == 400
assert not result.duplicated(
    ["audio_id", "hypothesis_id"]
).any()
assert result["word_count"].ge(0).all()
assert (
    result["unique_word_count"] <= result["word_count"]
).all()
assert result["type_token_ratio"].dropna().between(0, 1).all()
assert result["adjacent_repeat_ratio"].dropna().between(0, 1).all()

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUTPUT.with_suffix(".tmp")
result.to_csv(tmp, index=False, encoding="utf-8")

if OUTPUT.exists():
    assert sha256(OUTPUT) == sha256(tmp), (
        "Existing output differs; preserved for investigation."
    )
    tmp.unlink()
else:
    tmp.replace(OUTPUT)

# Measure linguistic feature variation for the same audio.
variation = result.groupby("audio_id")[feature_names].agg(
    lambda col: col.nunique(dropna=False)
)

variable_counts = {
    feature: int((variation[feature] > 1).sum())
    for feature in feature_names
}

report = {
    "status": "PASS",
    "feature_config_status": "proposed_not_frozen",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "feature_count": len(feature_names),
    "zero_word_transcripts": int(
        result["word_count"].eq(0).sum()
    ),
    "mean_word_count": float(result["word_count"].mean()),
    "feature_variable_audio_counts": variable_counts,
    "source_asr_sha256": EXPECTED_ASR_SHA,
    "output_sha256": sha256(OUTPUT),
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_linguistic_extraction"] = "completed_pending_review"
progress["step4_linguistic_features_sha256"] = sha256(OUTPUT)
progress["next_task"] = (
    "Review linguistic features and assess suitability "
    "for fluency prediction"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.1 LINGUISTIC EXTRACTION SUMMARY ===")
print("Status:", report["status"])
print("Audio count:", report["audio_count"])
print("Hypotheses:", report["hypothesis_count"])
print("Features:", report["feature_count"])
print("Mean word count:", round(report["mean_word_count"], 3))
print("Zero-word transcripts:", report["zero_word_transcripts"])

print("\n=== FEATURE VARIABILITY ACROSS H1-H5 ===")
for name, count in variable_counts.items():
    print(f"{name}: {count}/400 audios")

print("\n=== FEATURE DESCRIPTIVE STATISTICS ===")
print(
    result[feature_names]
    .describe()
    .round(4)
    .to_string()
)

print("\nCSV:", OUTPUT)
print("SHA256:", sha256(OUTPUT))
print("Configuration status:", feature_config["status"])
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURES = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

AUDIT = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
REPORT = PROJECT / "reports/step04_linguistic_anomaly_summary.json"

EXPECTED_ASR_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
EXPECTED_FEATURE_SHA = "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(ASR) == EXPECTED_ASR_SHA
assert sha256(FEATURES) == EXPECTED_FEATURE_SHA

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

keys = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]

assert len(asr) == len(features) == 2000
assert not asr.duplicated(keys).any()
assert not features.duplicated(keys).any()

merged = features.merge(
    asr[keys + ["transcription", "duration_sec"]],
    on=keys,
    how="inner",
    validate="one_to_one"
)

assert len(merged) == 2000

# Diagnostic thresholds only; not exclusion criteria.
merged["flag_long_transcript"] = merged["word_count"] >= 20
merged["flag_low_ttr"] = merged["type_token_ratio"] < 0.5
merged["flag_long_mean_word"] = merged["mean_word_length"] > 10
merged["flag_high_repetition"] = merged["adjacent_repeat_count"] >= 2

flags = [
    "flag_long_transcript",
    "flag_low_ttr",
    "flag_long_mean_word",
    "flag_high_repetition"
]

merged["any_flag"] = merged[flags].any(axis=1)

anomalies = merged.loc[merged["any_flag"]].copy()

# Preserve original transcriptions and all numeric values.
AUDIT.parent.mkdir(parents=True, exist_ok=True)

temp = AUDIT.with_suffix(".tmp")
anomalies.to_csv(temp, index=False, encoding="utf-8")

if AUDIT.exists():
    assert sha256(AUDIT) == sha256(temp), (
        "Existing anomaly audit differs; no overwrite."
    )
    temp.unlink()
else:
    temp.replace(AUDIT)

one_word_count = int(merged["word_count"].eq(1).sum())
missing_repeat_ratio = int(
    merged["adjacent_repeat_ratio"].isna().sum()
)

assert one_word_count == missing_repeat_ratio, (
    "Unexpected missing adjacent_repeat_ratio values."
)

summary = {
    "status": "generated_pending_review",
    "total_hypotheses": 2000,
    "flagged_hypotheses": int(len(anomalies)),
    "flagged_unique_audios": int(anomalies["audio_id"].nunique()),
    "long_transcripts_ge_20": int(merged["flag_long_transcript"].sum()),
    "low_ttr_lt_0_5": int(merged["flag_low_ttr"].sum()),
    "long_mean_word_gt_10": int(merged["flag_long_mean_word"].sum()),
    "high_repetition_ge_2": int(merged["flag_high_repetition"].sum()),
    "one_word_transcripts": one_word_count,
    "missing_repeat_ratio": missing_repeat_ratio,
    "source_asr_sha256": sha256(ASR),
    "source_features_sha256": sha256(FEATURES),
    "audit_csv_sha256": sha256(AUDIT),
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_linguistic_anomaly_audit"] = "generated_pending_review"
progress["step4_linguistic_anomaly_report"] = str(REPORT)
progress["next_task"] = (
    "Inspect extreme ASR transcripts before freezing linguistic features"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.2 ANOMALY SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== TOP 10 LONGEST TRANSCRIPTIONS ===")
longest = merged.sort_values(
    ["word_count", "pilot_index"],
    ascending=[False, True]
).head(10)

for _, row in longest.iterrows():
    print("\nAudio:", row["audio_id"],
          "| Hypothesis:", row["hypothesis_id"],
          "| Words:", row["word_count"],
          "| TTR:", round(row["type_token_ratio"], 4))
    print("Transcript:", repr(row["transcription"][:1000]))

print("\n=== TOP 10 LOWEST TTR TRANSCRIPTIONS ===")
lowest = merged.sort_values(
    ["type_token_ratio", "pilot_index"],
    ascending=[True, True]
).head(10)

for _, row in lowest.iterrows():
    print(
        row["audio_id"],
        row["hypothesis_id"],
        "words=", row["word_count"],
        "TTR=", round(row["type_token_ratio"], 4),
        "repeats=", row["adjacent_repeat_count"]
    )
    print("Transcript:", repr(row["transcription"][:500]))

print("\nAudit CSV:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURES = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

AUDIT = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
REPORT = PROJECT / "reports/step04_linguistic_anomaly_summary.json"

EXPECTED_ASR_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
EXPECTED_FEATURE_SHA = "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(ASR) == EXPECTED_ASR_SHA
assert sha256(FEATURES) == EXPECTED_FEATURE_SHA

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

numeric_cols = [
    "word_count",
    "unique_word_count",
    "type_token_ratio",
    "mean_word_length",
    "character_count",
    "adjacent_repeat_count",
    "adjacent_repeat_ratio"
]

# Convert CSV blanks into actual NaN values.
for col in numeric_cols:
    features[col] = pd.to_numeric(
        features[col],
        errors="coerce"
    )

keys = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]

assert len(asr) == len(features) == 2000
assert not asr.duplicated(keys).any()
assert not features.duplicated(keys).any()

merged = features.merge(
    asr[keys + ["transcription", "duration_sec"]],
    on=keys,
    how="inner",
    validate="one_to_one"
)

assert len(merged) == 2000

# Verify the missing-value policy.
one_word_count = int(merged["word_count"].eq(1).sum())
missing_repeat_ratio = int(
    merged["adjacent_repeat_ratio"].isna().sum()
)

assert one_word_count == missing_repeat_ratio, (
    f"Missing ratio mismatch: "
    f"one_word={one_word_count}, "
    f"missing={missing_repeat_ratio}"
)

assert merged["word_count"].notna().all()
assert merged["adjacent_repeat_count"].notna().all()

# Diagnostic flags only — no exclusions.
merged["flag_long_transcript"] = (
    merged["word_count"] >= 20
)
merged["flag_low_ttr"] = (
    merged["type_token_ratio"] < 0.5
)
merged["flag_long_mean_word"] = (
    merged["mean_word_length"] > 10
)
merged["flag_high_repetition"] = (
    merged["adjacent_repeat_count"] >= 2
)

flags = [
    "flag_long_transcript",
    "flag_low_ttr",
    "flag_long_mean_word",
    "flag_high_repetition"
]

merged["any_flag"] = merged[flags].any(axis=1)

anomalies = merged.loc[
    merged["any_flag"]
].copy()

AUDIT.parent.mkdir(parents=True, exist_ok=True)

temp = AUDIT.with_suffix(".tmp")
anomalies.to_csv(
    temp,
    index=False,
    encoding="utf-8"
)

# The earlier failed Cell 34 may already have created
# an audit CSV. Verify its content before reuse.
if AUDIT.exists():
    assert sha256(AUDIT) == sha256(temp), (
        "Existing anomaly audit differs. "
        "Original file preserved."
    )
    temp.unlink()
else:
    temp.replace(AUDIT)

summary = {
    "status": "generated_pending_review",
    "total_hypotheses": 2000,
    "flagged_hypotheses": int(len(anomalies)),
    "flagged_unique_audios": int(
        anomalies["audio_id"].nunique()
    ),
    "long_transcripts_ge_20": int(
        merged["flag_long_transcript"].sum()
    ),
    "low_ttr_lt_0_5": int(
        merged["flag_low_ttr"].sum()
    ),
    "long_mean_word_gt_10": int(
        merged["flag_long_mean_word"].sum()
    ),
    "high_repetition_ge_2": int(
        merged["flag_high_repetition"].sum()
    ),
    "one_word_transcripts": one_word_count,
    "missing_repeat_ratio": missing_repeat_ratio,
    "source_asr_sha256": sha256(ASR),
    "source_features_sha256": sha256(FEATURES),
    "audit_csv_sha256": sha256(AUDIT),
    "created_at": datetime.now(
        timezone.utc
    ).isoformat()
}

REPORT.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step4_linguistic_anomaly_audit"] = (
    "generated_pending_review"
)
progress["step4_linguistic_anomaly_report"] = str(REPORT)
progress["next_task"] = (
    "Inspect extreme ASR transcripts before "
    "freezing linguistic features"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== STEP 4.2 ANOMALY SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== TOP 10 LONGEST TRANSCRIPTIONS ===")
longest = merged.sort_values(
    ["word_count", "pilot_index"],
    ascending=[False, True]
).head(10)

for _, row in longest.iterrows():
    print(
        "\nAudio:", row["audio_id"],
        "| Hypothesis:", row["hypothesis_id"],
        "| Words:", row["word_count"],
        "| TTR:", round(row["type_token_ratio"], 4)
    )
    print(
        "Transcript:",
        repr(row["transcription"][:1000])
    )

print("\n=== TOP 10 LOWEST TTR TRANSCRIPTIONS ===")
lowest = merged.sort_values(
    ["type_token_ratio", "pilot_index"],
    ascending=[True, True]
).head(10)

for _, row in lowest.iterrows():
    print(
        row["audio_id"],
        row["hypothesis_id"],
        "words=", row["word_count"],
        "TTR=", round(row["type_token_ratio"], 4),
        "repeats=", row["adjacent_repeat_count"]
    )
    print(
        "Transcript:",
        repr(row["transcription"][:500])
    )

print("\n=== MISSING VALUE VALIDATION ===")
print("One-word transcripts:", one_word_count)
print("Missing adjacent_repeat_ratio:", missing_repeat_ratio)
print("Validation: PASS")
print("Audit CSV:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import numpy as np

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR_FILE = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURE_FILE = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
ANOMALY_FILE = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
TRAIN_FILE = PROJECT / "data/raw/speechocean762_train.parquet"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "reports/step04_targeted_anomaly_inspection.csv"

keys = [
    "pilot_index", "audio_id", "speaker", "hypothesis_id"
]

asr = pd.read_csv(
    ASR_FILE,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURE_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

anomalies = pd.read_csv(
    ANOMALY_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(asr) == len(features) == 2000
assert len(anomalies) == 13

flagged_ids = sorted(anomalies["audio_id"].unique())

assert len(flagged_ids) == 5
assert "030210164" in flagged_ids

merged = asr.merge(
    features[
        keys + [
            "word_count",
            "unique_word_count",
            "type_token_ratio",
            "mean_word_length",
            "adjacent_repeat_count"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

selected = merged.loc[
    merged["audio_id"].isin(flagged_ids)
].sort_values(["audio_id", "hypothesis_id"]).copy()

assert len(selected) == 25

# Retrieve reading prompts and audio for the flagged cases.
pf = pq.ParquetFile(TRAIN_FILE)

audio_info = {}

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id,
        columns=["audio", "text"]
    )

    audios = table.column("audio").to_pylist()
    prompts = table.column("text").to_pylist()

    for audio, prompt in zip(audios, prompts):
        audio_id = Path(audio["path"]).stem

        if audio_id in flagged_ids:
            assert audio_id not in audio_info
            audio_info[audio_id] = {
                "audio_bytes": audio["bytes"],
                "reading_prompt": str(prompt)
            }

assert set(audio_info) == set(flagged_ids)

selected["reading_prompt"] = selected["audio_id"].map(
    lambda aid: audio_info[aid]["reading_prompt"]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
selected.to_csv(OUT, index=False, encoding="utf-8")

print("=== STEP 4.2 TARGETED ANOMALY INSPECTION ===")
print("Flagged unique audios:", len(flagged_ids))
print("Hypotheses for inspection:", len(selected))

for aid in flagged_ids:
    subset = selected.loc[selected["audio_id"] == aid]

    print("\n" + "=" * 70)
    print("Audio ID:", aid)
    print("Reading prompt:",
          repr(audio_info[aid]["reading_prompt"]))

    for _, row in subset.iterrows():
        print(
            f"\n{row['hypothesis_id']} "
            f"| words={int(row['word_count'])} "
            f"| TTR={row['type_token_ratio']:.4f} "
            f"| mean_length={row['mean_word_length']:.2f}"
        )
        print("Transcript:", repr(row["transcription"]))

print("\n=== CRITICAL AUDIO PLAYBACK: 030210164 ===")

critical = audio_info["030210164"]
y, sr = sf.read(
    io.BytesIO(critical["audio_bytes"]),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1
assert np.isfinite(y).all()

print("Duration:", round(len(y) / sr, 3), "seconds")
print("Prompt:", repr(critical["reading_prompt"]))

display(Audio(y, rate=sr, normalize=False))

# Preserve the review as pending until listening is completed.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_targeted_anomaly_inspection"] = (
    "generated_pending_listening_review"
)
progress["step4_targeted_anomaly_file"] = str(OUT)
progress["next_task"] = (
    "Review audio 030210164 and classify ASR repetition anomaly"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== INSPECTION SAVED ===")
print("CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import numpy as np

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR_FILE = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURE_FILE = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
ANOMALY_FILE = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
TRAIN_FILE = PROJECT / "data/raw/speechocean762_train.parquet"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "reports/step04_targeted_anomaly_inspection.csv"

keys = [
    "pilot_index", "audio_id", "speaker", "hypothesis_id"
]

asr = pd.read_csv(
    ASR_FILE,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURE_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

anomalies = pd.read_csv(
    ANOMALY_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(asr) == len(features) == 2000
assert len(anomalies) == 13

flagged_ids = sorted(anomalies["audio_id"].unique())

assert len(flagged_ids) == 5
assert "030210164" in flagged_ids

merged = asr.merge(
    features[
        keys + [
            "word_count",
            "unique_word_count",
            "type_token_ratio",
            "mean_word_length",
            "adjacent_repeat_count"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

selected = merged.loc[
    merged["audio_id"].isin(flagged_ids)
].sort_values(["audio_id", "hypothesis_id"]).copy()

assert len(selected) == 25

# Retrieve reading prompts and audio for the flagged cases.
pf = pq.ParquetFile(TRAIN_FILE)

audio_info = {}

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id,
        columns=["audio", "text"]
    )

    audios = table.column("audio").to_pylist()
    prompts = table.column("text").to_pylist()

    for audio, prompt in zip(audios, prompts):
        audio_id = Path(audio["path"]).stem

        if audio_id in flagged_ids:
            assert audio_id not in audio_info
            audio_info[audio_id] = {
                "audio_bytes": audio["bytes"],
                "reading_prompt": str(prompt)
            }

assert set(audio_info) == set(flagged_ids)

selected["reading_prompt"] = selected["audio_id"].map(
    lambda aid: audio_info[aid]["reading_prompt"]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
selected.to_csv(OUT, index=False, encoding="utf-8")

print("=== STEP 4.2 TARGETED ANOMALY INSPECTION ===")
print("Flagged unique audios:", len(flagged_ids))
print("Hypotheses for inspection:", len(selected))

for aid in flagged_ids:
    subset = selected.loc[selected["audio_id"] == aid]

    print("\n" + "=" * 70)
    print("Audio ID:", aid)
    print("Reading prompt:",
          repr(audio_info[aid]["reading_prompt"]))

    for _, row in subset.iterrows():
        print(
            f"\n{row['hypothesis_id']} "
            f"| words={int(row['word_count'])} "
            f"| TTR={row['type_token_ratio']:.4f} "
            f"| mean_length={row['mean_word_length']:.2f}"
        )
        print("Transcript:", repr(row["transcription"]))

print("\n=== CRITICAL AUDIO PLAYBACK: 030210164 ===")

critical = audio_info["030210164"]
y, sr = sf.read(
    io.BytesIO(critical["audio_bytes"]),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1
assert np.isfinite(y).all()

print("Duration:", round(len(y) / sr, 3), "seconds")
print("Prompt:", repr(critical["reading_prompt"]))

display(Audio(y, rate=sr, normalize=False))

# Preserve the review as pending until listening is completed.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_targeted_anomaly_inspection"] = (
    "generated_pending_listening_review"
)
progress["step4_targeted_anomaly_file"] = str(OUT)
progress["next_task"] = (
    "Review audio 030210164 and classify ASR repetition anomaly"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== INSPECTION SAVED ===")
print("CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

INSPECTION = PROJECT / "reports/step04_targeted_anomaly_inspection.csv"
ANOMALIES = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "reports/step04_anomaly_classification_v1.csv"
REPORT = PROJECT / "reports/step04_anomaly_classification_v1.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

inspection = pd.read_csv(
    INSPECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

anomalies = pd.read_csv(
    ANOMALIES,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(inspection) == 25
assert len(anomalies) == 13

ids = sorted(inspection["audio_id"].unique())
assert len(ids) == 5

classification = {
    "030210137": {
        "category": "prompt_consistent_repetition",
        "evidence": "prompt_and_all_five_asr_hypotheses",
        "review_status": "audio_not_listened",
        "notes": (
            "Repeated 'care' matches reading prompt. "
            "Actual delivery not independently verified by listening."
        )
    },
    "030210164": {
        "category": "confirmed_asr_phrase_repetition_loop",
        "evidence": "user_audio_listening_plus_asr_hypotheses",
        "review_status": "user_listening_reviewed",
        "notes": (
            "User heard the target words with difficult accent. "
            "H2 repeated 'God bless you' 32 times, inconsistent "
            "with the audio. No verbatim human transcript certified."
        )
    },
    "096340009": {
        "category": "non_english_script_and_character_repetition",
        "evidence": "asr_text_inspection",
        "review_status": "audio_not_listened",
        "notes": (
            "Chinese-script output and character repetitions. "
            "Listening verification pending."
        )
    },
    "096340012": {
        "category": "mixed_script_and_repetitive_decoding",
        "evidence": "asr_text_inspection",
        "review_status": "audio_not_listened",
        "notes": (
            "Mixed English, romanized and Chinese characters. "
            "Listening verification pending."
        )
    },
    "096340017": {
        "category": "non_english_script_repetition_all_hypotheses",
        "evidence": "asr_text_inspection",
        "review_status": "audio_not_listened",
        "notes": (
            "All five hypotheses contain repeated Chinese-script "
            "characters. Listening verification pending."
        )
    }
}

assert set(ids) == set(classification)

rows = []

for aid in ids:
    subset = inspection.loc[inspection["audio_id"] == aid]
    item = classification[aid]

    rows.append({
        "audio_id": aid,
        "pilot_index": int(subset["pilot_index"].iloc[0]),
        "speaker": str(subset["speaker"].iloc[0]),
        "reading_prompt": str(subset["reading_prompt"].iloc[0]),
        "flagged_hypotheses": int(
            (anomalies["audio_id"] == aid).sum()
        ),
        "total_hypotheses": 5,
        **item,
        "primary_analysis_include": True,
        "sensitivity_analysis_candidate": (
            aid != "030210137"
        )
    })

result = pd.DataFrame(rows)

OUT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUT.with_suffix(".tmp")
result.to_csv(tmp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(tmp), (
        "Existing classification differs; original preserved."
    )
    tmp.unlink()
else:
    tmp.replace(OUT)

report = {
    "status": "preliminary_classification_saved",
    "audio_count": 5,
    "flagged_hypotheses": 13,
    "confirmed_phrase_repetition_loop_audio_count": 1,
    "audio_listening_pending_count": 4,
    "primary_analysis_excluded_audio_count": 0,
    "sensitivity_candidates": int(
        result["sensitivity_analysis_candidate"].sum()
    ),
    "classification_csv_sha256": sha256(OUT),
    "source_inspection_sha256": sha256(INSPECTION),
    "source_anomaly_sha256": sha256(ANOMALIES),
    "created_at": datetime.now(timezone.utc).isoformat()
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    for key in report:
        if key != "created_at":
            assert previous.get(key) == report[key]
else:
    tmp_report = REPORT.with_suffix(".tmp")
    tmp_report.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))

progress["step4_anomaly_classification"] = (
    "preliminary_saved"
)
progress["step4_anomaly_classification_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Review feature suitability and define robust "
    "ASR anomaly indicators before model training"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.3 ANOMALY CLASSIFICATION SAVED ===")
print(result[
    [
        "audio_id",
        "category",
        "review_status",
        "primary_analysis_include",
        "sensitivity_analysis_candidate"
    ]
].to_string(index=False))

print("\nConfirmed repetition-loop audios:",
      report["confirmed_phrase_repetition_loop_audio_count"])
print("Pending audio listening:",
      report["audio_listening_pending_count"])
print("Primary analysis exclusions:",
      report["primary_analysis_excluded_audio_count"])
print("Sensitivity candidates:",
      report["sensitivity_candidates"])
print("CSV:", OUT)
print("SHA256:", sha256(OUT))
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUTPUT = (
    PROJECT /
    "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
)
REPORT = (
    PROJECT /
    "reports/step04_prompt_asr_disagreement_pilot_40.json"
)

EXPECTED_ASR_SHA = (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(ASR) == EXPECTED_ASR_SHA
assert TRAIN.is_file()

# Use the same basic tokenizer as STEP 4.1.
TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

def tokens(text):
    return [
        word.casefold()
        for word in TOKEN_PATTERN.findall(str(text))
    ]

def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, item_a in enumerate(a, start=1):
        current = [i]

        for j, item_b in enumerate(b, start=1):
            current.append(
                min(
                    current[j - 1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (item_a != item_b)
                )
            )

        previous = current

    return previous[-1]

def normalized_words(text):
    return tokens(text)

def normalized_chars(text):
    return " ".join(tokens(text))

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index")

assert len(asr) == 2000
assert len(pilot) == 400

# Deterministic pilot: 10 recordings from each 100-audio batch.
# These are chosen without using human fluency labels.
selected_parts = []

for start in [1, 101, 201, 301]:
    block = pilot.loc[
        pilot["pilot_index"].between(start, start + 99)
    ]

    selected_parts.append(
        block.iloc[np.linspace(0, 99, 10, dtype=int)]
    )

selected = pd.concat(
    selected_parts,
    ignore_index=True
)

assert len(selected) == 40
assert selected["audio_id"].nunique() == 40

wanted = set(selected["audio_id"].astype(str))
prompts = {}

pf = pq.ParquetFile(TRAIN)

assert "audio" in pf.schema_arrow.names
assert "text" in pf.schema_arrow.names

for rg in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        rg,
        columns=["audio", "text"]
    )

    audio_rows = table.column("audio").to_pylist()
    text_rows = table.column("text").to_pylist()

    for audio, prompt in zip(audio_rows, text_rows):
        aid = Path(audio["path"]).stem

        if aid in wanted:
            assert aid not in prompts
            prompts[aid] = str(prompt)

assert set(prompts) == wanted

subset = asr.loc[
    asr["audio_id"].isin(wanted)
].copy()

assert len(subset) == 200

records = []

for _, row in subset.iterrows():
    aid = str(row["audio_id"])
    prompt = prompts[aid]
    transcript = str(row["transcription"])

    p_words = normalized_words(prompt)
    h_words = normalized_words(transcript)

    p_chars = normalized_chars(prompt)
    h_chars = normalized_chars(transcript)

    assert len(p_words) > 0
    assert len(p_chars) > 0

    word_distance = edit_distance(p_words, h_words)
    char_distance = edit_distance(p_chars, h_chars)

    records.append({
        "pilot_index": int(row["pilot_index"]),
        "audio_id": aid,
        "speaker": str(row["speaker"]),
        "hypothesis_id": row["hypothesis_id"],
        "reading_prompt": prompt,
        "transcription": transcript,
        "prompt_word_count": len(p_words),
        "asr_word_count": len(h_words),
        "prompt_asr_word_edit_distance": word_distance,
        "prompt_asr_word_disagreement": (
            word_distance / len(p_words)
        ),
        "prompt_asr_char_edit_distance": char_distance,
        "prompt_asr_char_disagreement": (
            char_distance / len(p_chars)
        ),
        "prompt_asr_exact_match": (
            p_words == h_words
        )
    })

result = pd.DataFrame(records).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(result) == 200
assert result["audio_id"].nunique() == 40
assert not result.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

temp = OUTPUT.with_suffix(".tmp")
result.to_csv(temp, index=False, encoding="utf-8")

if OUTPUT.exists():
    assert sha256(OUTPUT) == sha256(temp)
    temp.unlink()
else:
    temp.replace(OUTPUT)

summary = {
    "status": "pilot_completed_pending_review",
    "audio_count": 40,
    "hypothesis_count": 200,
    "mean_prompt_asr_word_disagreement": float(
        result["prompt_asr_word_disagreement"].mean()
    ),
    "median_prompt_asr_word_disagreement": float(
        result["prompt_asr_word_disagreement"].median()
    ),
    "mean_prompt_asr_char_disagreement": float(
        result["prompt_asr_char_disagreement"].mean()
    ),
    "exact_match_hypotheses": int(
        result["prompt_asr_exact_match"].sum()
    ),
    "source_asr_sha256": sha256(ASR),
    "output_sha256": sha256(OUTPUT),
    "interpretation": (
        "Prompt-ASR disagreement, not validated WER/CER "
        "against a verbatim human transcript."
    ),
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    for key, value in summary.items():
        if key != "created_at":
            assert previous.get(key) == value
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_prompt_asr_pilot"] = "completed_pending_review"
progress["step4_prompt_asr_pilot_sha256"] = sha256(OUTPUT)
progress["next_task"] = (
    "Review prompt-ASR disagreement pilot before "
    "deciding whether to include prompt alignment features"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.4 PROMPT-ASR PILOT SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== DISAGREEMENT DISTRIBUTION ===")
print(
    result[
        [
            "prompt_asr_word_disagreement",
            "prompt_asr_char_disagreement"
        ]
    ].describe().round(4).to_string()
)

print("\n=== TOP 10 HIGHEST DISAGREEMENTS ===")
top = result.sort_values(
    "prompt_asr_word_disagreement",
    ascending=False
).head(10)

for _, row in top.iterrows():
    print(
        "\nAudio:", row["audio_id"],
        "| H:", row["hypothesis_id"],
        "| word disagreement:",
        round(row["prompt_asr_word_disagreement"], 4)
    )
    print("Prompt:", repr(row["reading_prompt"]))
    print("ASR:", repr(row["transcription"][:300]))

print("\nCSV:", OUTPUT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = (
    PROJECT /
    "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
)
REPORT = (
    PROJECT /
    "reports/step04_prompt_asr_robustness_pilot_40.json"
)
PER_AUDIO = (
    PROJECT /
    "reports/step04_prompt_asr_robustness_per_audio_40.csv"
)
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED_SHA = (
    "bff38db4e4b5f1ee711091d0a621aac42a8214e1ad10fab9d18f6b2c29794143"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_SHA

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 200
assert df["audio_id"].nunique() == 40
assert df["hypothesis_id"].value_counts().to_dict() == {
    h: 40 for h in ["H1", "H2", "H3", "H4", "H5"]
}

WORD = "prompt_asr_word_disagreement"
CHAR = "prompt_asr_char_disagreement"

assert df[[WORD, CHAR]].notna().all().all()
assert (df[[WORD, CHAR]] >= 0).all().all()

# The previously confirmed phrase repetition-loop case.
outlier_mask = (
    df["audio_id"].eq("030210164") &
    df["hypothesis_id"].eq("H2")
)

assert int(outlier_mask.sum()) == 1
assert np.isclose(
    df.loc[outlier_mask, WORD].iloc[0], 16.0
)

def descriptive_stats(series):
    s = pd.to_numeric(series, errors="raise")
    return {
        "n": int(len(s)),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
        "iqr": float(s.quantile(0.75) - s.quantile(0.25)),
        "p95": float(s.quantile(0.95)),
        "max": float(s.max())
    }

overall_word = descriptive_stats(df[WORD])
overall_char = descriptive_stats(df[CHAR])

# Sensitivity calculation only; original data is retained.
without_one = df.loc[~outlier_mask].copy()

without_outlier_word = descriptive_stats(without_one[WORD])
without_outlier_char = descriptive_stats(without_one[CHAR])

by_hypothesis = {}

for hid in ["H1", "H2", "H3", "H4", "H5"]:
    subset = df.loc[df["hypothesis_id"] == hid]

    by_hypothesis[hid] = {
        "word": descriptive_stats(subset[WORD]),
        "character": descriptive_stats(subset[CHAR]),
        "exact_matches": int(
            subset["prompt_asr_exact_match"]
            .astype(str).str.lower().eq("true").sum()
        )
    }

# Each audio has five hypotheses. Use this to measure
# within-audio disagreement variability.
per_audio = (
    df.groupby(["audio_id", "speaker", "pilot_index"])
    .agg(
        word_mean=(WORD, "mean"),
        word_min=(WORD, "min"),
        word_max=(WORD, "max"),
        word_std=(WORD, "std"),
        char_mean=(CHAR, "mean"),
        char_min=(CHAR, "min"),
        char_max=(CHAR, "max"),
        char_std=(CHAR, "std")
    )
    .reset_index()
)

per_audio["word_range"] = (
    per_audio["word_max"] - per_audio["word_min"]
)
per_audio["char_range"] = (
    per_audio["char_max"] - per_audio["char_min"]
)

assert len(per_audio) == 40

variable_word_audios = int(
    (per_audio["word_range"] > 1e-12).sum()
)
variable_char_audios = int(
    (per_audio["char_range"] > 1e-12).sum()
)

REPORT.parent.mkdir(parents=True, exist_ok=True)

tmp_csv = PER_AUDIO.with_suffix(".tmp")
per_audio.to_csv(tmp_csv, index=False, encoding="utf-8")

if PER_AUDIO.exists():
    assert sha256(PER_AUDIO) == sha256(tmp_csv), (
        "Existing per-audio output differs."
    )
    tmp_csv.unlink()
else:
    tmp_csv.replace(PER_AUDIO)

report = {
    "status": "robustness_pilot_complete",
    "source_sha256": EXPECTED_SHA,
    "audio_count": 40,
    "hypothesis_count": 200,
    "word_disagreement_all": overall_word,
    "char_disagreement_all": overall_char,
    "outlier_audio_id": "030210164",
    "outlier_hypothesis_id": "H2",
    "word_disagreement_without_one_outlier":
        without_outlier_word,
    "char_disagreement_without_one_outlier":
        without_outlier_char,
    "by_hypothesis": by_hypothesis,
    "audios_with_variable_word_disagreement":
        variable_word_audios,
    "audios_with_variable_char_disagreement":
        variable_char_audios,
    "per_audio_csv_sha256": sha256(PER_AUDIO),
    "interpretation": (
        "Prompt-ASR disagreement, not ground-truth WER/CER. "
        "Outlier removal is diagnostic only, not a dataset edit."
    ),
    "created_at": datetime.now(timezone.utc).isoformat()
}

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    for key, value in report.items():
        if key != "created_at":
            assert previous.get(key) == value
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step4_prompt_asr_robustness"] = (
    "pilot_complete_pending_review"
)
progress["step4_prompt_asr_robustness_report"] = str(REPORT)
progress["next_task"] = (
    "Review robust prompt-ASR measures and "
    "finalize linguistic feature design"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== STEP 4.5 ROBUSTNESS PILOT SUMMARY ===")

for name, stats in [
    ("WORD ALL", overall_word),
    ("WORD WITHOUT ONE OUTLIER", without_outlier_word),
    ("CHAR ALL", overall_char),
    ("CHAR WITHOUT ONE OUTLIER", without_outlier_char)
]:
    print("\n", name)
    for key in ["n", "mean", "median", "iqr", "p95", "max"]:
        print(f"{key}: {stats[key]:.4f}")

print("\n=== H1-H5 COMPARISON ===")
for hid, stats in by_hypothesis.items():
    print(
        hid,
        "word_mean=", round(stats["word"]["mean"], 4),
        "word_median=", round(stats["word"]["median"], 4),
        "char_mean=", round(stats["character"]["mean"], 4),
        "exact_matches=", stats["exact_matches"]
    )

print("\n=== WITHIN-AUDIO VARIABILITY ===")
print("Audios with variable word disagreement:",
      variable_word_audios)
print("Audios with variable char disagreement:",
      variable_char_audios)

print("\nReport:", REPORT)
print("Per-audio CSV:", PER_AUDIO)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
LING = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
ACOUSTIC = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
PROMPT_PILOT = (
    PROJECT / "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
)
ROBUSTNESS = (
    PROJECT / "reports/step04_prompt_asr_robustness_pilot_40.json"
)
CONFIG = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED = {
    str(ASR): "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80",
    str(LING): "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459",
    str(ACOUSTIC): "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced",
    str(PROMPT_PILOT): "bff38db4e4b5f1ee711091d0a621aac42a8214e1ad10fab9d18f6b2c29794143"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in EXPECTED.items():
    path = Path(filename)
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected_hash

robustness = json.loads(
    ROBUSTNESS.read_text(encoding="utf-8")
)

assert robustness["status"] == "robustness_pilot_complete"
assert robustness["audio_count"] == 40
assert robustness["hypothesis_count"] == 200

linguistic_features = [
    "word_count",
    "unique_word_count",
    "type_token_ratio",
    "mean_word_length",
    "character_count",
    "adjacent_repeat_count",
    "adjacent_repeat_ratio"
]

prompt_features = [
    "prompt_asr_word_disagreement",
    "prompt_asr_char_disagreement"
]

model_specs = {
    "A": {
        "description": "Acoustic-only baseline",
        "feature_groups": ["acoustic"]
    },
    "B1": {
        "description": "Transcript-only baseline",
        "feature_groups": ["linguistic"]
    },
    "B2": {
        "description": "Prompt-aware transcript ablation",
        "feature_groups": ["linguistic", "prompt"]
    },
    "C1": {
        "description": "Acoustic + transcript fusion",
        "feature_groups": ["acoustic", "linguistic"]
    },
    "C2": {
        "description": "Acoustic + prompt-aware fusion",
        "feature_groups": ["acoustic", "linguistic", "prompt"]
    }
}

protocol = {
    "name": "fluency_feature_design_v1",
    "status": "design_predeclared_pending_full_prompt_extraction",
    "dataset": "mispeech/speechocean762",
    "selected_audio_count": 400,
    "asr_hypothesis_count": 2000,
    "target": "human_fluency_score",
    "acoustic_feature_count": 14,
    "acoustic_feature_names": (
        "read exact 14 column names from STEP 2 artifact"
    ),
    "linguistic_feature_names": linguistic_features,
    "prompt_feature_names": prompt_features,
    "models": model_specs,
    "primary_models": ["A", "B1", "C1"],
    "secondary_ablation_models": ["B2", "C2"],
    "validation_rules": {
        "speaker_disjoint": True,
        "same_audio_hypotheses_same_partition": True,
        "preprocessing_fit_on_training_only": True,
        "hyperparameter_selection_within_training_data": True,
        "human_labels_not_used_for_feature_selection_now": True,
        "sealed_official_test_split_untouched": True
    },
    "missing_value_policy": {
        "adjacent_repeat_ratio": (
            "NaN for one-word transcripts; "
            "impute using training-fold statistics only"
        ),
        "other_features": (
            "check and handle inside training folds"
        )
    },
    "scientific_cautions": [
        "Reading prompt is not a verbatim human transcript.",
        "Prompt-ASR disagreement is not ground-truth WER or CER.",
        "Transcript lexical diversity does not directly measure vocabulary proficiency.",
        "Transcript repetition may reflect ASR errors rather than speech disfluency.",
        "ASR anomalies remain in primary analysis.",
        "Sensitivity exclusions require explicit documented rules.",
        "Prompt-aware gains may reflect access to target sentence information."
    ],
    "next_required_task": (
        "Extract two prompt-ASR disagreement features "
        "for all 400 audios and 2000 hypotheses"
    ),
    "source_sha256": {
        "asr": sha256(ASR),
        "linguistic": sha256(LING),
        "acoustic": sha256(ACOUSTIC),
        "prompt_pilot": sha256(PROMPT_PILOT)
    }
}

CONFIG.parent.mkdir(parents=True, exist_ok=True)

if CONFIG.exists():
    previous = json.loads(CONFIG.read_text(encoding="utf-8"))
    assert previous == protocol, (
        "Feature design config differs; existing file preserved."
    )
else:
    temp = CONFIG.with_suffix(".tmp")
    temp.write_text(
        json.dumps(protocol, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(CONFIG)

progress = json.loads(CHECK.read_text(encoding="utf-8"))

progress["step4_feature_design"] = "predeclared"
progress["step4_feature_design_sha256"] = sha256(CONFIG)
progress["next_task"] = protocol["next_required_task"]
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.6 FEATURE DESIGN SAVED ===")
print("Status:", protocol["status"])
print("Primary models:", protocol["primary_models"])
print("Ablation models:", protocol["secondary_ablation_models"])
print("Acoustic features:", protocol["acoustic_feature_count"])
print("Linguistic features:", len(linguistic_features))
print("Prompt features:", len(prompt_features))
print("Speaker-disjoint validation: REQUIRED")
print("Sealed official test: UNTOUCHED")
print("Design config:", CONFIG)
print("SHA256:", sha256(CONFIG))
print("Next task:", progress["next_task"])


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
PILOT40 = PROJECT / "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
REPORT = PROJECT / "reports/step04_prompt_asr_full_0400_qc.json"

EXPECTED_ASR_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
EXPECTED_DESIGN_SHA = "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
EXPECTED_PILOT_SHA = "bff38db4e4b5f1ee711091d0a621aac42a8214e1ad10fab9d18f6b2c29794143"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected_sha in [
    (ASR, EXPECTED_ASR_SHA),
    (DESIGN, EXPECTED_DESIGN_SHA),
    (PILOT40, EXPECTED_PILOT_SHA)
]:
    assert path.is_file(), f"Missing file: {path}"
    assert sha256(path) == expected_sha, f"SHA mismatch: {path}"

assert TRAIN.is_file()
assert PILOT.is_file()

design = json.loads(DESIGN.read_text(encoding="utf-8"))
assert design["selected_audio_count"] == 400
assert design["asr_hypothesis_count"] == 2000
assert design["prompt_feature_names"] == [
    "prompt_asr_word_disagreement",
    "prompt_asr_char_disagreement"
]

TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

def tokens(text):
    return [
        word.casefold()
        for word in TOKEN_PATTERN.findall(str(text))
    ]

def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, item_a in enumerate(a, start=1):
        current = [i]

        for j, item_b in enumerate(b, start=1):
            current.append(
                min(
                    current[j - 1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (item_a != item_b)
                )
            )

        previous = current

    return previous[-1]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index")

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(pilot) == 400
assert pilot["pilot_index"].tolist() == list(range(1, 401))
assert pilot["audio_id"].is_unique
assert pilot["audio_id"].notna().all()
assert pilot["speaker"].notna().all()

assert len(asr) == 2000
assert asr["audio_id"].nunique() == 400
assert asr["pilot_index"].nunique() == 400
assert asr["transcription"].str.strip().ne("").all()

keys = ["pilot_index", "audio_id", "speaker"]

verified = asr.merge(
    pilot[keys],
    on=keys,
    how="inner",
    validate="many_to_one"
)

assert len(verified) == 2000
assert not verified.duplicated(
    keys + ["hypothesis_id"]
).any()

HIDS = ["H1", "H2", "H3", "H4", "H5"]

assert verified.groupby("audio_id")["hypothesis_id"].apply(
    lambda s: sorted(s.tolist()) == HIDS
).all()

# Retrieve only prompts, not decoded audio.
wanted = set(pilot["audio_id"].astype(str))
prompts = {}

pf = pq.ParquetFile(TRAIN)

assert "audio" in pf.schema_arrow.names
assert "text" in pf.schema_arrow.names

for rg in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        rg,
        columns=["audio", "text"]
    )

    audio_rows = table.column("audio").to_pylist()
    text_rows = table.column("text").to_pylist()

    for audio, prompt in zip(audio_rows, text_rows):
        aid = Path(audio["path"]).stem

        if aid in wanted:
            assert aid not in prompts, f"Duplicate prompt: {aid}"
            prompts[aid] = str(prompt)

assert set(prompts) == wanted, (
    f"Missing prompts: {sorted(wanted - set(prompts))[:10]}"
)

print("=== STEP 4.7 FULL EXTRACTION START ===")
print("Audios:", len(prompts))
print("Hypotheses:", len(verified))
print("Feature design SHA256:", sha256(DESIGN))

records = []

for _, row in verified.iterrows():
    aid = str(row["audio_id"])
    prompt = prompts[aid]
    transcript = str(row["transcription"])

    p_words = tokens(prompt)
    h_words = tokens(transcript)

    p_chars = " ".join(p_words)
    h_chars = " ".join(h_words)

    assert len(p_words) > 0, f"Empty prompt tokens: {aid}"
    assert len(p_chars) > 0

    word_distance = edit_distance(p_words, h_words)
    char_distance = edit_distance(p_chars, h_chars)

    records.append({
        "pilot_index": int(row["pilot_index"]),
        "audio_id": aid,
        "speaker": str(row["speaker"]),
        "hypothesis_id": str(row["hypothesis_id"]),
        "reading_prompt": prompt,
        "transcription": transcript,
        "prompt_word_count": len(p_words),
        "asr_word_count": len(h_words),
        "prompt_asr_word_edit_distance": word_distance,
        "prompt_asr_word_disagreement": (
            word_distance / len(p_words)
        ),
        "prompt_asr_char_edit_distance": char_distance,
        "prompt_asr_char_disagreement": (
            char_distance / len(p_chars)
        ),
        "prompt_asr_exact_match": p_words == h_words
    })

result = pd.DataFrame(records).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(result) == 2000
assert result["audio_id"].nunique() == 400
assert not result.duplicated(
    keys + ["hypothesis_id"]
).any()

WORD = "prompt_asr_word_disagreement"
CHAR = "prompt_asr_char_disagreement"

assert result[[WORD, CHAR]].notna().all().all()
assert (result[[WORD, CHAR]] >= 0).all().all()

# Verify 40-audio pilot results match exactly in
# identifiers, prompts, transcriptions, and metrics.
old = pd.read_csv(
    PILOT40,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(old) == 200

compare_keys = keys + ["hypothesis_id"]

matched = old.merge(
    result,
    on=compare_keys,
    suffixes=("_pilot", "_full"),
    how="inner",
    validate="one_to_one"
)

assert len(matched) == 200

for col in ["reading_prompt", "transcription"]:
    assert (
        matched[f"{col}_pilot"] ==
        matched[f"{col}_full"]
    ).all(), f"Pilot mismatch: {col}"

metric_columns = [
    "prompt_word_count",
    "asr_word_count",
    "prompt_asr_word_edit_distance",
    WORD,
    "prompt_asr_char_edit_distance",
    CHAR
]

for col in metric_columns:
    a = pd.to_numeric(matched[f"{col}_pilot"])
    b = pd.to_numeric(matched[f"{col}_full"])

    assert np.allclose(a, b, rtol=0, atol=1e-12), (
        f"Pilot metric mismatch: {col}"
    )

assert (
    matched["prompt_asr_exact_match_pilot"].astype(str).str.lower()
    ==
    matched["prompt_asr_exact_match_full"].astype(str).str.lower()
).all()

# Save without overwriting differing previous results.
OUT.parent.mkdir(parents=True, exist_ok=True)

temp = OUT.with_suffix(".tmp")
result.to_csv(temp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(temp), (
        "Existing full prompt output differs; original preserved."
    )
    temp.unlink()
else:
    temp.replace(OUT)

# Within-audio feature variability.
variation = result.groupby("audio_id").agg(
    word_range=(WORD, lambda s: float(s.max() - s.min())),
    char_range=(CHAR, lambda s: float(s.max() - s.min()))
)

summary = {
    "status": "PASS",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "feature_count": 2,
    "pilot_40_consistency": "PASS",
    "mean_word_disagreement": float(result[WORD].mean()),
    "median_word_disagreement": float(result[WORD].median()),
    "mean_char_disagreement": float(result[CHAR].mean()),
    "median_char_disagreement": float(result[CHAR].median()),
    "exact_match_hypotheses": int(
        result["prompt_asr_exact_match"].sum()
    ),
    "audios_with_variable_word_disagreement": int(
        (variation["word_range"] > 1e-12).sum()
    ),
    "audios_with_variable_char_disagreement": int(
        (variation["char_range"] > 1e-12).sum()
    ),
    "max_word_disagreement": float(result[WORD].max()),
    "max_char_disagreement": float(result[CHAR].max()),
    "source_asr_sha256": EXPECTED_ASR_SHA,
    "design_sha256": EXPECTED_DESIGN_SHA,
    "output_sha256": sha256(OUT),
    "interpretation": (
        "Reading prompt versus ASR output disagreement; "
        "not ground-truth WER or CER."
    )
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == summary, (
        "Existing QC report differs; original preserved."
    )
else:
    tmp_report = REPORT.with_suffix(".tmp")
    tmp_report.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_prompt_asr_full"] = "qc_pass"
progress["step4_prompt_asr_full_sha256"] = sha256(OUT)
progress["step4_prompt_asr_pilot_consistency"] = "PASS"
progress["next_task"] = (
    "Review full prompt-ASR features, merge feature groups "
    "and prepare speaker-disjoint modeling"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== STEP 4.7 FULL PROMPT-ASR SUMMARY ===")
for name, value in summary.items():
    print(f"{name}: {value}")

print("\n=== DISAGREEMENT DISTRIBUTION ===")
print(
    result[[WORD, CHAR]]
    .describe()
    .round(4)
    .to_string()
)

print("\n=== H1-H5 COMPARISON ===")
print(
    result.groupby("hypothesis_id")[[WORD, CHAR]]
    .agg(["mean", "median"])
    .round(4)
    .to_string()
)

print("\n=== PER-BATCH COMPARISON ===")
result["batch"] = (
    ((result["pilot_index"] - 1) // 100) + 1
)
print(
    result.groupby("batch")[[WORD, CHAR]]
    .agg(["mean", "median"])
    .round(4)
    .to_string()
)

print("\nCSV:", OUT)
print("SHA256:", sha256(OUT))
print("QC report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ACOUSTIC = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
LINGUISTIC = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
PROMPT = PROJECT / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

REPORT = PROJECT / "reports/step04_modeling_schema_audit.json"

EXPECTED = {
    ACOUSTIC: "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced",
    LINGUISTIC: "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459",
    PROMPT: "8b52805a91144c5ee0a379aeb2954595f55ec085203ac38861845c5fbe33c25c",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path.name}"

def read_csv(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

acoustic = read_csv(ACOUSTIC)
linguistic = read_csv(LINGUISTIC)
prompt = read_csv(PROMPT)

assert len(acoustic) == 400
assert len(linguistic) == 2000
assert len(prompt) == 2000

assert acoustic["audio_id"].nunique() == 400
assert linguistic["audio_id"].nunique() == 400
assert prompt["audio_id"].nunique() == 400

assert not acoustic["audio_id"].duplicated().any()

KEYS = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]

assert all(c in linguistic.columns for c in KEYS)
assert all(c in prompt.columns for c in KEYS)

assert not linguistic.duplicated(KEYS).any()
assert not prompt.duplicated(KEYS).any()

lk = linguistic[KEYS].sort_values(KEYS).reset_index(drop=True)
pk = prompt[KEYS].sort_values(KEYS).reset_index(drop=True)

assert lk.equals(pk), "Linguistic/prompt key mismatch"

assert set(acoustic["audio_id"]) == set(linguistic["audio_id"])

# Check identity mapping without assuming acoustic schema.
identity_cols = [
    c for c in ["pilot_index", "audio_id", "speaker"]
    if c in acoustic.columns
]

identity = linguistic[
    ["pilot_index", "audio_id", "speaker"]
].drop_duplicates()

matched = acoustic[identity_cols].merge(
    identity,
    on="audio_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_acoustic", "_reference")
)

assert len(matched) == 400

for c in ["pilot_index", "speaker"]:
    if c in acoustic.columns:
        left = matched[f"{c}_acoustic"].astype(str)
        right = matched[f"{c}_reference"].astype(str)
        assert left.equals(right), f"{c} identity mismatch"

design = json.loads(DESIGN.read_text(encoding="utf-8"))

ling_features = design["linguistic_feature_names"]
prompt_features = design["prompt_feature_names"]

assert len(ling_features) == 7
assert len(prompt_features) == 2
assert all(c in linguistic.columns for c in ling_features)
assert all(c in prompt.columns for c in prompt_features)

# Inspect available acoustic columns without guessing
# which are predictors or labels.
schema = {
    "status": "PASS_pending_acoustic_feature_selection",
    "acoustic_rows": len(acoustic),
    "linguistic_rows": len(linguistic),
    "prompt_rows": len(prompt),
    "unique_audio_count": 400,
    "speaker_count": int(linguistic["speaker"].nunique()),
    "acoustic_columns": acoustic.columns.tolist(),
    "acoustic_dtypes": {
        c: str(acoustic[c].dtype)
        for c in acoustic.columns
    },
    "acoustic_missing_counts": {
        c: int(acoustic[c].isna().sum())
        for c in acoustic.columns
    },
    "linguistic_features": ling_features,
    "prompt_features": prompt_features,
    "identity_check": "PASS",
    "hypothesis_key_check": "PASS",
    "source_sha256": {
        str(p.relative_to(PROJECT)): sha256(p)
        for p in EXPECTED
    },
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    existing = json.loads(REPORT.read_text(encoding="utf-8"))
    for key, value in schema.items():
        if key != "created_at":
            assert existing.get(key) == value, (
                f"Existing schema report mismatch: {key}"
            )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(schema, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_modeling_schema_audit"] = "pass_pending_feature_mapping"
progress["next_task"] = (
    "Confirm 14 acoustic feature columns and human "
    "fluency label mapping before merging modeling dataset"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.8 MODELING SCHEMA AUDIT ===")
print("Status:", schema["status"])
print("Acoustic rows:", len(acoustic))
print("Linguistic rows:", len(linguistic))
print("Prompt rows:", len(prompt))
print("Unique audios:", schema["unique_audio_count"])
print("Speakers:", schema["speaker_count"])
print("Identity check: PASS")
print("Hypothesis key check: PASS")

print("\n=== ACOUSTIC COLUMNS AND DTYPES ===")
for col in acoustic.columns:
    print(
        f"{col} | {acoustic[col].dtype} "
        f"| missing={acoustic[col].isna().sum()}"
    )

print("\n=== ACOUSTIC SAMPLE ===")
print(acoustic.head(3).to_string(index=False))

print("\n=== LINGUISTIC FEATURE NAMES ===")
print(ling_features)

print("\n=== PROMPT FEATURE NAMES ===")
print(prompt_features)

print("\nSchema report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ACOUSTIC = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
LING = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
PROMPT = PROJECT / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
REPORT = PROJECT / "reports/step04_modeling_dataset_qc.json"

EXPECTED = {
    ACOUSTIC: "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced",
    LING: "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459",
    PROMPT: "8b52805a91144c5ee0a379aeb2954595f55ec085203ac38861845c5fbe33c25c",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

def read_csv(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

acoustic = read_csv(ACOUSTIC)
ling = read_csv(LING)
prompt = read_csv(PROMPT)

design = json.loads(DESIGN.read_text(encoding="utf-8"))

ids = ["pilot_index", "audio_id", "speaker"]
keys = ids + ["hypothesis_id"]

ling_features = design["linguistic_feature_names"]
prompt_features = design["prompt_feature_names"]

# Explicitly exclude target and identifiers.
acoustic_features = [
    c for c in acoustic.columns
    if c not in ids + ["fluency"]
]

assert len(acoustic_features) == 14
assert len(ling_features) == 7
assert len(prompt_features) == 2

all_features = (
    acoustic_features + ling_features + prompt_features
)

assert len(all_features) == 23
assert len(set(all_features)) == 23
assert "fluency" not in all_features

assert len(acoustic) == 400
assert len(ling) == 2000
assert len(prompt) == 2000

assert acoustic["audio_id"].is_unique
assert not acoustic.duplicated(ids).any()
assert not ling.duplicated(keys).any()
assert not prompt.duplicated(keys).any()

assert set(acoustic["audio_id"]) == set(ling["audio_id"])
assert set(ling["audio_id"]) == set(prompt["audio_id"])

# Select only approved feature columns.
base = acoustic[
    ids + ["fluency"] + acoustic_features
].copy()

ling_data = ling[
    keys + ling_features
].copy()

prompt_data = prompt[
    keys + prompt_features
].copy()

# Strict one-to-one merge for hypothesis-level features.
hyp = ling_data.merge(
    prompt_data,
    on=keys,
    how="inner",
    validate="one_to_one"
)

assert len(hyp) == 2000

# Each audio has five hypotheses.
modeling = hyp.merge(
    base,
    on=ids,
    how="inner",
    validate="many_to_one"
)

assert len(modeling) == 2000
assert modeling["audio_id"].nunique() == 400
assert modeling["speaker"].nunique() == 20
assert not modeling.duplicated(keys).any()

assert modeling.groupby("audio_id")["hypothesis_id"].apply(
    lambda s: sorted(s.tolist()) == [
        "H1", "H2", "H3", "H4", "H5"
    ]
).all()

# Verify the target is constant within an audio.
assert modeling.groupby("audio_id")["fluency"].nunique().eq(1).all()
assert modeling["fluency"].notna().all()

# Compare every target value against its original acoustic row.
target_check = modeling[
    ids + ["fluency"]
].drop_duplicates()

assert len(target_check) == 400

original_targets = base[
    ids + ["fluency"]
].sort_values(ids).reset_index(drop=True)

merged_targets = target_check.sort_values(
    ids
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    original_targets,
    merged_targets,
    check_dtype=False
)

# All predictors must be numeric.
for col in all_features:
    assert pd.api.types.is_numeric_dtype(modeling[col]), (
        f"Non-numeric predictor: {col}"
    )

numeric = modeling[all_features].to_numpy(dtype=float)
assert not np.isinf(numeric).any()

# Keep missing values for training-fold-only imputation.
missing = {
    col: int(modeling[col].isna().sum())
    for col in all_features
    if modeling[col].isna().sum() > 0
}

# Final explicit column order.
ordered_columns = (
    keys + ["fluency"] +
    acoustic_features +
    ling_features +
    prompt_features
)

modeling = modeling[
    ordered_columns
].sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert modeling.shape == (2000, 28)

# Verify every acoustic feature remains identical across H1-H5.
for col in acoustic_features:
    counts = modeling.groupby("audio_id")[col].nunique(
        dropna=False
    )
    assert counts.eq(1).all(), (
        f"Acoustic feature differs across hypotheses: {col}"
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

temp = OUT.with_suffix(".tmp")
modeling.to_csv(
    temp,
    index=False,
    encoding="utf-8"
)

if OUT.exists():
    assert sha256(OUT) == sha256(temp), (
        "Existing modeling dataset differs; original preserved."
    )
    temp.unlink()
else:
    temp.replace(OUT)

report = {
    "status": "PASS",
    "rows": 2000,
    "columns": 28,
    "unique_audios": 400,
    "unique_speakers": 20,
    "hypotheses_per_audio": 5,
    "target": "fluency",
    "acoustic_features": acoustic_features,
    "linguistic_features": ling_features,
    "prompt_features": prompt_features,
    "total_predictors": 23,
    "missing_predictor_values": missing,
    "target_integrity": "PASS",
    "acoustic_invariance": "PASS",
    "source_sha256": {
        str(p.relative_to(PROJECT)): sha256(p)
        for p in EXPECTED
    },
    "output_sha256": sha256(OUT)
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert previous == report, (
        "Existing QC report differs; original preserved."
    )
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_modeling_dataset"] = "qc_pass"
progress["step4_modeling_dataset_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Audit human fluency label distribution and speaker groups "
    "before constructing speaker-disjoint validation splits"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.9 MODELING DATASET QC ===")
print("Status:", report["status"])
print("Rows:", report["rows"])
print("Columns:", report["columns"])
print("Unique audios:", report["unique_audios"])
print("Unique speakers:", report["unique_speakers"])
print("Target:", report["target"])
print("Acoustic features:", len(acoustic_features))
print("Linguistic features:", len(ling_features))
print("Prompt features:", len(prompt_features))
print("Total predictors:", report["total_predictors"])
print("Target integrity:", report["target_integrity"])
print("Acoustic invariance:", report["acoustic_invariance"])

print("\n=== MISSING PREDICTOR VALUES ===")
print(missing)

print("\n=== HUMAN FLUENCY DISTRIBUTION (400 UNIQUE AUDIOS) ===")
print(
    modeling.drop_duplicates("audio_id")["fluency"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== MODELING DATASET SAMPLE ===")
print(
    modeling[
        keys + [
            "fluency",
            "duration_sec",
            "word_count",
            "prompt_asr_word_disagreement"
        ]
    ].head(10).to_string(index=False)
)

print("\nCSV:", OUT)
print("SHA256:", sha256(OUT))
print("QC report:", REPORT)
print("Checkpoint updated.")
